# 콩당콩당 RAG 시스템 평가 파이프라인 (RAGAS)

본 노트북은 멀티에이전트 RAG 구조의 핵심적 성능 지표인 **신뢰성(Faithfulness)**, **답변 적합성(Answer Relevance)**, **컨텍스트 재현율(Context Recall)**, **컨텍스트 정밀도(Context Precision)**를 RAGAS 프레임워크를 활용하여 정량 측정 및 리포트하는 평가 도구 코드입니다.

In [ ]:
# 1. RAGAS 패키지 설치
!pip install -q ragas datasets pandas openai

In [ ]:
# 2. 임포트 및 Ragas 데이터셋 구성
import os
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevance,
    context_recall,
    context_precision
)

# OpenAI API Key 설정 필요
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "your_openai_api_key")

In [ ]:
# 3. 평가용 골든 데이터셋(Q-A-Context GroundTruth) 로드
# 포맷: {'question': List[str], 'contexts': List[List[str]], 'answer': List[str], 'ground_truth': List[str]}
eval_data = {
    "question": [
        "만성 신부전 3기 환자 칼륨 제한량은?",
        "당뇨병 환자 미세알부민 검사는 주기와 목적이 무엇인가요?"
    ],
    "contexts": [
        ["만성 신부전 3기 환자의 경우 소변을 통한 칼륨 조절 기능이 급격히 저하되므로, 하루 2,000mg 이하의 섭취량을 유지해야 합니다."],
        ["당뇨 환자들은 당뇨병성 신증 합병증을 조기에 차단하기 위해 연 1회 이상의 요미세알부민뇨 측정을 지침으로 삼고 있습니다."]
    ],
    "answer": [
        "만성 신부전 3기 환자는 칼륨 조절 장애를 예방하기 위해 소금 및 칼륨이 포함된 신선 채소류를 주의 깊게 전처리하여 2000mg 미만으로 제한 섭취해야 합니다.",
        "당뇨 환자의 미세알부민 검사 주기는 연 1회 이상이며, 주된 목적은 조기에 신장 합병증 사구체 손상 징후를 판독하기 위함입니다."
    ],
    "ground_truth": [
        "만성 신부전 환자는 사구체 여과율 감소에 따라 고칼륨혈증 발생을 억제하기 위해 일일 칼륨 섭취량을 2,000mg 이하로 조절합니다.",
        "당뇨 환자의 신장 사구체 모세혈관 누출 손상을 조기 진단하기 위해 일 년에 최소 한 번 이상 미세알부민뇨 정량 검사를 시행하도록 명시합니다."
    ]
}

dataset = Dataset.from_dict(eval_data)
print("RAGAS 평가용 데이터셋 로드 완료!")

In [ ]:
# 4. Ragas 평가 수행 및 결과 대시보드 리포팅
metrics = [faithfulness, answer_relevance, context_recall, context_precision]
result = evaluate(dataset, metrics=metrics)

df_res = result.to_pandas()
print("\n--- [RAGAS 평가 최종 스코어 요약] ---")
print(df_res.mean(numeric_only=True))

# 결과를 CSV로 저장
df_res.to_csv("ragas_evaluation_results.csv", index=False)